# 1. Business Understanding: Pemetaan Tutupan Lahan Provinsi Jawa Timur Berbasis Citra Satelit Sentinel-2

**Mata Kuliah**: Proyek Sains Data  
**Kerangka Kerja**: Cross-Industry Standard Process for Data Mining (CRISP-DM)  
**Wilayah Studi**: Provinsi Jawa Timur, Indonesia  
**Sensor Citra**: Satelit Multi-Spektral Copernicus Sentinel-2 Level-2A (Bottom-of-Atmosphere Reflectance)  

---

## 1.1 Latar Belakang dan Urgensi Pemantauan Geospasial

Provinsi Jawa Timur merupakan salah satu pusat pertumbuhan ekonomi, industri, dan lumbung pangan strategis nasional. Wilayah ini memiliki karakteristik bentang lahan yang sangat dinamis dan heterogen, mulai dari aglomerasi perkotaan metropolitan (seperti Surabaya Raya dan koridor Gerbangkertosusila), hamparan sawah irigasi teknis di sepanjang cekungan Sungai Brantas, kawasan hutan lindung dataran tinggi di gugusan pegunungan (Arjuno-Welirang, Semeru, Ijen), hingga ekosistem hutan mangrove di pesisir utara (Gresik, Lamongan, Probolinggo) dan pesisir selatan (Taman Nasional Baluran dan Alas Purwo).

Perubahan tutupan lahan (*Land Use / Land Cover - LULC*) yang berlangsung cepat—terutama konversi lahan pertanian basah menjadi kawasan industri dan permukiman (*urban sprawl*) serta degradasi kawasan pesisir—menuntut ketersediaan sistem pemantauan berkala yang cepat, akurat, dan berbiaya efisien. Survei terestrial secara manual di lapangan memerlukan waktu yang panjang dan biaya logistik yang tinggi. Oleh karena itu, penerapan teknologi penginderaan jauh (*remote sensing*) berbasis satelit multi-spektral resolusi menengah-tinggi (Sentinel-2 10m - 20m) yang dipadukan dengan algoritma pembelajaran mesin (*machine learning*) menjadi pendekatan mutakhir yang tepat sasaran.

---

## 1.2 Definisi dan Sasaran 5 Kelas Tutupan Lahan

Sesuai dengan spesifikasi penugasan dan standar pemetaan tematik Badan Informasi Geospasial (BIG), proyek ini memetakan 5 kelas tutupan lahan representatif di Jawa Timur:

1. **Bangunan (*Built-up Area*)**: Mencakup permukiman padat perkotaan, gedung komersial, perumahan perdesaan, kawasan industri manufaktur, dan infrastruktur transportasi (jalan aspal/beton).
2. **Lahan Pertanian (*Farmland / Agricultural Land*)**: Mencakup sawah irigasi teknis, tegalan, ladang palawija, sawah berair (fase penggenangan), dan petak sawah bera/kosongan pasca-panen.
3. **Perairan (*Water Body*)**: Mencakup perairan terbuka murni permanen, seperti waduk/danau (Waduk Sutami, Selorejo), aliran sungai utama, muara, dan perairan laut pesisir.
4. **Hutan Daratan (*Terrestrial Forest*)**: Mencakup hutan hujan pegunungan, hutan lindung, perkebunan pohon kayu keras (seperti jati Perhutani), dan vegetasi kanopi pohon berkayu lebat di daratan kering berdrainase baik.
5. **Hutan Mangrove (*Mangrove Forest*)**: Mencakup formasi vegetasi pohon bakau pesisir di zona pasang-surut (*intertidal zone*) yang berakar di atas substrat sedimen lumpur basah berair asin/payau.

---

## 1.3 Paradigma Klasifikasi: Strategi Separasi Dua Arah Mutual (*Mutual Disambiguation*) Menggunakan 5 Indeks Biofisik Terarah

Tantangan terbesar dalam klasifikasi spektral multi-kelas bukanlah sekadar mengenali ciri suatu kelas, melainkan **mencegah kesalahan tumpang tindih dua arah (*bi-directional confusion*)**: kelas target tidak boleh disangka sebagai kelas lain (*False Negative* dicegah), dan sebaliknya kelas lain tidak boleh keliru terbaca sebagai kelas target (*False Positive* dicegah).

Untuk menyelesaikan ambiguitas spektral tersebut, sistem ini mengintegrasikan **5 Indeks Biofisik Terarah Fiks** yang masing-masing memiliki mandat fisis eksplisit:

```
+---------------------------------------------------------------------------------------------------------+
|                       ARSITEKTUR SEPARASI DUA ARAH 5 INDEKS BIOFISIK SENTINEL-2                         |
+=========================+=============================+=================================================+
| Indeks Spektral         | Kelas yang Ditingkatkan     | Kelas yang Diredam / Ditolak (Mutual Exclusion) |
+=========================+=============================+=================================================+
| 1. REMI                 | Hutan Mangrove              | Meredam Hutan Darat & Lahan Terbuka             |
| 2. NDVI                 | Hutan Darat & Kanopi Rapat  | Meredam Non-Vegetasi; Kombinasi REMI vs Mangrove|
| 3. IBI                  | Bangunan (Built-up)         | Meredam Tanah Kosong & Sawah Kering Gersang     |
| 4. MNDWI                | Sawah Berair / Lembap       | Membedakan Sawah Basah vs Perairan Terbuka Murni|
| 5. NDWI                 | Perairan Terbuka Murni      | Menarik garis batas air tegas; tolak daratan/rawa|
+-------------------------+-----------------------------+-------------------------------------------------+
```

### Rincian Peran & Rumus Spektral Fiks Masing-Masing Indeks:

#### 1. Perhitungan REMI (*Red Edge Mangrove Index*) $\rightarrow$ Pemisahan Mangrove vs Hutan Daratan
* **Mandat Bisnis**: Menonjolkan kelas **Hutan Mangrove** agar tidak tertukar dengan **Hutan Daratan Biasa**, dan sebaliknya.
* **Rumus Fiks Sentinel-2**:
  $$\mathbf{REMI = \frac{B8A - B04}{(B11 - B03) + \epsilon}}$$
  *Keterangan Parameter Komputasi:*
  * **$B8A$ (*Narrow NIR / Red Edge*, 865 nm)**: Saluran inframerah sempit bebas gangguan serapan uap air laut; menangkap pantulan kanopi mangrove secara murni.
  * **$B04$ (*Red*, 665 nm)**: Saluran merah penyerap klorofil fotosintesis daun.
  * **$B11$ (*SWIR-1*, 1610 nm)**: Saluran inframerah gelombang pendek yang terserap sangat kuat oleh substrat lumpur dan air payau intertidal.
  * **$B03$ (*Green*, 560 nm)**: Saluran hijau penyeimbang spektrum latar belakang air dangkal.
  * **$\epsilon = 10^{-6}$**: Konstanta penstabil numerik untuk mencegah galat pembagian dengan nol (*zero division*).
* **Fisika Spektral**: Mangrove dan Hutan Darat sama-sama memiliki kanopi hijau lebat dengan nilai klorofil tinggi. Namun, habitat mangrove berada di zona intertidal dengan substrat lumpur jenuh air laut/payau yang menyerap spektrum SWIR-1 ($B11$) secara drastis, sementara pita *Red-Edge* ($B8A$) tetap memantulkan energi kanopi secara optimal. Selisih $(B11 - B03)$ pada penyebut menjadi sangat kecil pada mangrove, mendongkrak nilai **REMI menjadi sangat tinggi**.
* **Mekanisme Dua Arah**: 
  * Pada Mangrove: Pantulan Red Edge tinggi dikombinasikan dengan serapan kuat SWIR-1 menghasilkan nilai **REMI yang sangat tinggi**.
  * Pada Hutan Darat: Karena substrat tanah kering berdrainase baik, pantulan SWIR-1 jauh lebih tinggi daripada mangrove, sehingga nilai **REMI hutan darat relatif rendah/sedang**.
  * *Hasil*: Hutan daratan tidak akan keliru diklasifikasikan sebagai mangrove, dan mangrove terlindungi dari salah klasifikasi sebagai hutan biasa.

#### 2. Perhitungan NDVI (*Normalized Difference Vegetation Index*) $\rightarrow$ Kerapatan Vegetasi Darat & Sinergi REMI
* **Mandat Bisnis**: Mengisolasi kelas **Hutan Biasa & Vegetasi Rapat**, sekaligus bermitra dengan REMI dalam memisahkan hutan daratan versus mangrove.
* **Rumus Fiks Sentinel-2**:
  $$\mathbf{NDVI = \frac{B08 - B04}{B08 + B04 + \epsilon}}$$
* **Fisika Spektral**: Memanfaatkan kontras tajam antara serapan klorofil pada pita Red ($B04$) dan hamburan sel daun mesofil pada pita Broad NIR ($B08$).
* **Mekanisme Dua Arah**:
  * Objek non-vegetasi (air, bangunan, tanah terbuka) ditekan secara mutlak dengan nilai NDVI rendah ($< 0.2$) atau negatif.
  * Antara Hutan Darat vs Mangrove: Keduanya memiliki nilai NDVI tinggi ($> 0.6$). Model pembelajaran mesin memadukan nilai **NDVI tinggi + REMI rendah $\rightarrow$ Hutan Daratan**, sedangkan **NDVI tinggi + REMI tinggi $\rightarrow$ Mangrove**.

#### 3. Perhitungan IBI (*Index-based Built-up Index*) $\rightarrow$ Menonjolkan Bangunan & Meredam Tanah Kosong
* **Mandat Bisnis**: Menonjolkan kelas **Bangunan (*Built-up*)** dan secara aktif meredam wilayah **tanah kosong atau petak sawah kering gersang**.
* **Rumus Fiks Sentinel-2**:
  $$\mathbf{IBI = \frac{\text{NDBI} - \frac{\text{SAVI} + \text{MNDWI}}{2}}{\text{NDBI} + \frac{\text{SAVI} + \text{MNDWI}}{2} + \epsilon}}$$
  *dengan sub-indeks pembangun:*
  * $\text{NDBI} = \frac{B11 - B08}{B11 + B08 + \epsilon}$
  * $\text{SAVI} = \frac{(B08 - B04) \times (1 + 0.5)}{B08 + B04 + 0.5 + \epsilon}$
  * $\text{MNDWI} = \frac{B03 - B11}{B03 + B11 + \epsilon}$
* **Fisika Spektral**: Kelemahan fatal indeks bangunan konvensional (seperti NDBI murni) adalah sering rancu antara atap bangunan perkotaan dan tanah kosong berdebu karena keduanya memantulkan gelombang SWIR tinggi. IBI memecahkan masalah ini dengan meredam respon tanah latar belakang menggunakan pengurangan simultan indeks vegetasi tereliminasi tanah (SAVI) dan indeks air (MNDWI).
* **Mekanisme Dua Arah**:
  * Bangunan buatan manusia memiliki kombinasi pantulan spektral yang menghasilkan **IBI bernilai positif tinggi** ($> 0.05$).
  * Tanah kosong, tegalan gersang, dan sawah kering mengalami peredaman ganda sehingga nilai IBI tertekan menjadi netral atau negatif.
  * *Hasil*: Menghilangkan fenomena *false alarm* bangunan di area pertanian terbuka dan memastikan permukiman tidak tertukar dengan lahan kosong.

#### 4. Perhitungan MNDWI (*Modified Normalized Difference Water Index*) $\rightarrow$ Sawah Berair & Kelengasan Tanah
* **Mandat Bisnis**: Mendeteksi **Sawah Berair** serta membedakan gradasi kelembapan tanah pada **Sawah Kosongan / Bera**.
* **Rumus Fiks Sentinel-2**:
  $$\mathbf{MNDWI = \frac{B03 - B11}{B03 + B11 + \epsilon}}$$
* **Fisika Spektral**: Menggantikan saluran NIR dengan SWIR-1 ($B11$) untuk mendeteksi air permukaan secara lebih sensitif tanpa bias kekeruhan.
* **Mekanisme Dua Arah**:
  * Sawah berair / fase macak-macak memiliki kadar air tinggi bercampur lumpur, menghasilkan nilai **MNDWI positif moderat ($0.05 \text{ s/d } 0.35$)**.
  * Sawah kosongan yang lembap memiliki MNDWI mendekati nol, sedangkan sawah kering bernilai negatif tajam ($< -0.20$).
  * Perbedaan dengan Perairan Murni: Sawah berair memiliki nilai MNDWI yang lebih rendah dibanding perairan murni dan masih memiliki pantulan vegetasi/tanah, sehingga tidak tertukar dengan danau atau sungai.

#### 5. Perhitungan NDWI (*Normalized Difference Water Index*) $\rightarrow$ Deliniasi Perairan Murni Berbatas Tegas
* **Mandat Bisnis**: Menonjolkan kelas **Perairan Murni** (sungai, danau, waduk, laut) dengan menarik garis batas air (*waterline*) yang tegas dan tidak tertembus genangan pertanian.
* **Rumus Fiks Sentinel-2**:
  $$\mathbf{NDWI = \frac{B03 - B08}{B03 + B08 + \epsilon}}$$
* **Fisika Spektral**: Diformulasikan oleh McFeeters menggunakan saluran Green ($B03$) dan NIR ($B08$).
* **Mekanisme Dua Arah**:
  * Badan air terbuka murni menyerap radiasi NIR hampir sempurna, menghasilkan nilai **NDWI positif tinggi ($> 0.20$)**.
  * Seluruh objek daratan (vegetasi kanopi, tanaman padi, bangunan, dan tanah kering) memantulkan radiasi NIR secara kuat, menghasilkan nilai **NDWI negatif tajam ($< -0.10$)**.
  * *Hasil*: Garis pantai, tepian waduk, dan bantaran sungai terpisah secara presisi dari tambak dangkal, sawah irigasi, maupun mangrove tepi laut.

### Matriks Respon Spektral Separasi Dua Arah (*Bi-directional Disambiguation Matrix*)

| Kelas Tutupan Lahan | REMI | NDVI | IBI | MNDWI | NDWI | Profil Karakteristik Pengenal Kunci |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **1. Bangunan (*Built-up*)** | Rendah | Rendah | **Tinggi (+) Positif** | Negatif | Negatif Tajam | IBI mendominasi positif; menolak tanah kosong & sawah kering. |
| **2. Pertanian (*Farmland/Sawah*)** | Rendah-Sedang | Sedang-Tinggi | Negatif/Netral | **Moderat (+)** | Negatif | MNDWI mencerminkan kelembapan tanah; IBI teredam. |
| **3. Perairan (*Water Body*)** | Sangat Rendah | Negatif Tajam | Negatif Tajam | Tinggi (+) | **Sangat Tinggi (+) Positif** | NDWI mencapai puncak tertinggi; batas air sangat tegas. |
| **4. Hutan Daratan (*Forest*)** | Rendah-Sedang | **Sangat Tinggi (+)** | Negatif Tajam | Negatif Tajam | Negatif Tajam | NDVI puncak, namun REMI rendah karena substrat kering. |
| **5. Hutan Mangrove (*Mangrove*)** | **Sangat Tinggi (+)** | **Tinggi (+)** | Negatif Tajam | Sedang-Tinggi | Negatif | REMI tinggi membedakannya dari hutan daratan secara mutlak. |

---

## 1.4 Konfigurasi 7 Band Spektral Sentinel-2 sebagai Pondasi Fitur

Kelima indeks terarah di atas diturunkan secara langsung dari 7 band reflektansi utama Sentinel-2 Level-2A tanpa memerlukan data sensor eksternal:

| Pita Spektral Sentinel-2 | Panjang Gelombang | Resolusi Spasial | Kontribusi Fisis dalam Pembentukan 5 Indeks Terarah |
| :--- | :---: | :---: | :--- |
| **B02 (Blue)** | 490 nm | 10 meter | Penetrasi kolom air jernih dan koreksi pantulan atmosfer/tanah pada formula SAVI. |
| **B03 (Green)** | 560 nm | 10 meter | Puncak pantulan klorofil; komponen pembilang pada **NDWI** dan **MNDWI**, penyebut **REMI**. |
| **B04 (Red)** | 665 nm | 10 meter | Serapan kuat klorofil; komponen inti pengurang pada **NDVI**, **SAVI**, dan pembilang **REMI**. |
| **B08 (Broad NIR)** | 842 nm | 10 meter | Hamburan sel mesofil daun lebat; komponen pembeda utama vegetasi pada **NDVI** dan peredam pada **NDWI**. |
| **B8A (Narrow NIR)** | 865 nm | 20 meter | Inframerah sempit bebas uap air; menangkap sinyal kanopi basah mangrove pada formulasi **REMI**. |
| **B11 (SWIR-1)** | 1610 nm | 20 meter | Kunci kelembapan dan kekedapan; penyerap air pada **MNDWI** & **REMI**, pemantul material atap pada **IBI (NDBI)**. |
| **B12 (SWIR-2)** | 2190 nm | 20 meter | Membedakan mineral silika tanah pertanian kering dari material bangunan kedap air perkotaan. |

---

## 1.5 Alur Metodologi CRISP-DM Komprehensif (Tahap 1 hingga Tahap 6 Deployment)

Alur kerja proyek sains data ini dirancang secara sistematis mengikuti kerangka kerja standar industri CRISP-DM dari hulu ke hilir:

![Diagram Alur Kerja Terpadu Metodologi CRISP-DM](crisp_dm_workflow.png)

### 1. Tahap 1: Business Understanding
* **Tujuan**: Menentukan rumusan masalah spasial, mendefinisikan 5 kelas target tutupan lahan, menetapkan paradigma separasi dua arah (*mutual disambiguation*) dengan **5 Indeks Biofisik Terarah (REMI, NDVI, IBI, MNDWI, NDWI)**, serta menetapkan kriteria keberhasilan proyek yang terukur (standar BIG $\ge 80\%$).
* **Luaran**: Dokumen perumusan masalah, spesifikasi teknis kelas, matriks separasi dua arah, sasaran analitik, dan tolok ukur evaluasi.

### 2. Tahap 2: Data Understanding
* **Tujuan**: Memahami karakteristik data citra satelit dan memvalidasi respon spektral 5 indeks terarah pada poligon sampel lapangan (*ground truth*).
* **Kegiatan**:
  * Akuisisi data citra satelit Copernicus Sentinel-2 Level-2A (Bottom-of-Atmosphere Reflectance) bebas awan (< 5%) berukuran $1024 \times 1024$ piksel yang mencakup Surabaya, Sidoarjo, dan pesisir Gresik.
  * Digitasi 250 poligon sampel representatif (50 Bangunan, 50 Pertanian, 50 Perairan, 50 Hutan, 50 Mangrove) di Jawa Timur dalam format GeoJSON.
  * Ekstraksi **10.537 piksel spektral** dari 7 band dasar (B02, B03, B04, B08, B8A, B11, B12).
  * Eksplorasi data spasial (EDA): analisis tanda tangan spektral (*spectral signatures*), distribusi nilai 5 indeks baru per kelas (boxplot separasi), dan verifikasi mutual exclusion.
  * Peta interaktif 2D Leaflet pada notebook untuk memvalidasi sebaran poligon sampel beserta popup informasi 5 indeks terarah.
* **Luaran**: Dataset tabular sampel poligon (`samples_ground_truth.csv`), dataset piksel (`samples_extracted_pixels.csv`), dan visualisasi respon spektral.

### 3. Tahap 3: Data Preparation
* **Tujuan**: Mentransformasikan data mentah menjadi representasi fitur berkualitas tinggi yang siap diproses oleh model *machine learning*.
* **Kegiatan**:
  * Audit integritas data: konfirmasi 0 missing values, 0 infinity, 0 duplikat, dan batas reflektansi fisik $[0.0, 1.0]$.
  * Rekayasa Fitur Spektral Terarah (*Feature Engineering*): menghitung **5 Indeks Biofisik Terarah (REMI, NDVI, IBI, MNDWI, NDWI)** dari 7 band dasar, membentuk representasi **12 Fitur Prediktor Inti (7 Band BOA + 5 Indeks)**.
  * Pembagian dataset berbasis poligon (*Spatial Group Stratified 80:20 Split*): mempartisi data berbasis grup poligon (200 poligon latih dan 50 poligon uji independen) untuk menghasilkan **8.429 piksel data latih** dan **2.108 piksel data uji independen** bebas kebocoran spasial (*spatial data leakage*).
  * Penskalaan fitur (*Feature Scaling*): standardisasi fitur menggunakan `StandardScaler` yang di-fit hanya pada data latih dan disimpan sebagai artefak serialisasi.
* **Luaran**: Dataset piksel bersih `train_pixels.csv` (8.429 baris), `test_pixels.csv` (2.108 baris), `pixels_prepared_all.csv` (10.537 baris), dataset poligon `train_data.csv`, `test_data.csv`, serta artefak `scaler.pkl` & `scaler_pixels.pkl`.

### 4. Tahap 4: Modeling
* **Tujuan**: Membangun, melatih, dan mengoptimasi arsitektur algoritma *decision tree ensemble* berbasis piksel dengan regulasi ketat pencegah *overfitting*.
* **Kegiatan**:
  * Pelatihan model **LightGBM Pixel Classifier** dengan konfigurasi regularisasi anti-overfitting:
    * `max_depth = 6`, `num_leaves = 20` (membatasi kompleksitas pohon agar tidak menghafal noise).
    * `min_child_samples = 35` (menjamin setiap daun didukung sampel memadai).
    * `subsample = 0.8`, `colsample_bytree = 0.8` (stochastic subsampling per pohon).
    * `reg_alpha = 0.2`, `reg_lambda = 1.5` (penalti L1 dan L2 regularisasi bobot).
  * Validasi silang spasial (Stratified 5-Fold Cross Validation): mencatat akurasi piksel stabil **95.12% ± 0.55%** dan akurasi poligon **79.88% ± 1.82%**.
  * Analisis kepentingan fitur (*Feature Importance*): membuktikan dominasi peran REMI, NDVI, dan IBI dalam mengunci batas antar kelas.
* **Luaran**: Artefak model `model_lightgbm_piksel.pkl`, `metadata_lightgbm_piksel.json`, `model_lightgbm_teroptimasi.pkl`, dan `metadata_lightgbm_teroptimasi.json`.

### 5. Tahap 5: Evaluation
* **Tujuan**: Menguji kemampuan generalisasi model secara independen pada data uji (*unseen test set* 2.108 piksel) yang poligonnya belum pernah dilihat sama sekali oleh model saat training.
* **Kegiatan**:
  * Evaluasi metrik komprehensif pada data uji independen:
    * **Overall Accuracy (OA)**: **95.26%** (Melampaui standar minimal BIG 80%).
    * **Balanced Accuracy (BA)**: **95.05%** (Membuktikan performa seimbang di seluruh 5 kelas).
    * **Cohen's Kappa ($\kappa$)**: **0.9404** (Kategori *Almost Perfect Agreement*).
    * **Macro F1-Score**: **95.09%**.
    * **Macro ROC-AUC**: **99.68%**.
    * **Konsensus Poligon Spasial Holdout**: **99.14%** (43 dari 44 poligon uji terklasifikasi sempurna).
  * Analisis Matriks Konfusi (*Confusion Matrix*): membuktikan minimnya salah klasifikasi antar kelas:
    * Hutan Mangrove: Recall **95.65%**, Precision **93.09%** (REMI sukses meredam hutan daratan).
    * Perairan: Recall **99.74%**, Precision **100.00%** (NDWI sukses mengunci batas air tegas).
    * Bangunan: Recall **97.56%**, Precision **98.43%** (IBI sukses mengeliminasi sawah gersang).
    * Pertanian: Recall **94.40%**, Precision **94.40%**.
    * Hutan Daratan: Recall **90.03%**, Precision **89.51%**.
* **Luaran**: Laporan evaluasi komparatif dan artefak `metrics_evaluation.json` yang tersinkronisasi ke deployment.

### 6. Tahap 6: Deployment (Dua Jalur: Lokal & Online Cloud)
* **Tujuan**: Mengintegrasikan sistem klasifikasi ke dalam aplikasi dashboard Web GIS interaktif yang dapat diakses secara lokal maupun daring oleh pengambil kebijakan.
* **Arsitektur Dua Jalur Deployment**:
  1. **Jalur Deployment Lokal (`src/Tugas_UTS/6-deployment/`)**:
     * Notebook inferensi batch [`6-deployment.ipynb`](file:///d:/university/semester%205_book/proyek%20sain%20data/src/Tugas_UTS/6-deployment/6-deployment.ipynb): melakukan inferensi penuh pada $1.048.576$ piksel GeoTIFF scene dalam 17.73 detik, mengekspor hasil ke format GIS standar (`classified_lulc.tif` dan `confidence.tif`).
     * Aplikasi web interaktif [`app.py`](file:///d:/university/semester%205_book/proyek%20sain%20data/src/Tugas_UTS/6-deployment/app.py): menghubungkan pengguna akhir dengan peta tematik interaktif (Leaflet), inspektur spektral real-time, dan kalkulator indeks biofisik.
     * **4 Pilihan Mode Tampilan & Resolusi Kisi Piksel**:
       * **Mode Diperhalus (Filter Spasial seperti Awal)**: Mengaktifkan filter mayoritas spasial $3 \times 3$ dan rendering bilinear antialiasing sehingga zonasi tutupan lahan terlihat mulus, bersih dari efek bintik (*salt-and-pepper*).
       * **10 Meter (Kotak Piksel Asli Sentinel-2)**: Menampilkan resolusi murni sensor satelit ($10\,\text{m} \times 10\,\text{m}$) dengan grid tajam (*crisp-edges*).
       * **5 Meter (Super-Resolusi 2x Sub-Piksel)**: Menghasilkan kisi $4\times$ lebih rapat ($5\,\text{m} \times 5\,\text{m}$) untuk memetakan batas pematang sawah, jalan, dan tambak.
       * **2.5 Meter (Ultra-Detail 4x Sub-Piksel)**: Menghasilkan kisi $16\times$ lebih rapat ($2.5\,\text{m} \times 2.5\,\text{m}$) untuk analisis mikro batas tepi sungai dan vegetasi mangrove pesisir.
     * **Optimasi Kinerja**: Menggunakan engine *fast PIL bilinear interpolation* (waktu komputasi $< 0.2$ detik) dan *dynamic stride Leaflet overlay* agar aplikasi berjalan kilat tanpa hambatan (*freeze/lag*).
  2. **Jalur Deployment Online Cloud (`standalone_streamlit_app/`)**:
     * Repositori mandiri siap pasang pada **Streamlit Community Cloud** ([GitHub Repo](https://github.com/IlhamInsight/sentinel2-jatim-lulc-app)).
     * Seluruh artefak model (`models/`), data hasil persiapan (`prepared_data/`), batas poligon (`sampling_geojson/`), dan konfigurasi dependensi (`requirements.txt`) dikemas mandiri (*self-contained*) dengan mekanisme fallback path otomatis.
* **Luaran**: Sistem Web GIS siap pakai yang dapat dioperasikan secara lokal (`streamlit run src/Tugas_UTS/6-deployment/app.py`) dan daring melalui browser.

---

## 1.6 Kriteria Keberhasilan Proyek (*Success Criteria*)

Proyek ini menetapkan dua pilar kriteria keberhasilan yang selaras dengan paradigma separasi dua arah dan telah divalidasi secara empiris pada data uji independen:

| Indikator Performa | Target Awal (Ambang BIG / USGS) | Realisasi Hasil Pengujian Data Uji | Status Kepatuhan |
| :--- | :---: | :---: | :---: |
| **Overall Accuracy (OA)** | $\ge 80.00\%$ | **95.26%** | **Terpenuhi Unggul (+15.26%)** |
| **Balanced Accuracy (BA)** | $\ge 80.00\%$ | **95.05%** | **Terpenuhi Unggul (+15.05%)** |
| **Cohen's Kappa ($\kappa$)** | $\ge 0.7500$ | **0.9404** | **Terpenuhi Unggul (Almost Perfect)** |
| **Macro F1-Score** | $\ge 80.00\%$ | **95.09%** | **Terpenuhi Unggul (+15.09%)** |
| **Macro ROC-AUC** | $\ge 90.00\%$ | **99.68%** | **Terpenuhi Unggul (+9.68%)** |
| **Konsensus Poligon Spasial** | $\ge 85.00\%$ | **99.14%** | **Terpenuhi Unggul (+14.14%)** |
| **Recall Hutan Mangrove (REMI)** | $\ge 90.00\%$ | **95.65%** | **Terpenuhi (0 tumpang tindih hutan darat)** |
| **Precision Bangunan (IBI)** | $\ge 85.00\%$ | **98.43%** | **Terpenuhi (0 false alarm tanah gersang)** |
| **Recall Perairan Murni (NDWI)** | $\ge 95.00\%$ | **99.74%** | **Terpenuhi (batas air presisi tinggi)** |

---

## 1.7 Spesifikasi Lingkungan Komputasi dan Pustaka (*Technology Stack*)

Seluruh alur pemrosesan dijalankan secara terprogram murni menggunakan ekosistem terbuka bahasa pemrograman **Python** dan **Web GIS Leaflet**, tanpa ketergantungan pada perangkat lunak GIS desktop eksternal berbayar:

| Kategori Kebutuhan | Komponen / Pustaka | Versi Lingkungan | Peran & Fungsi Utama dalam Proyek |
| :--- | :--- | :---: | :--- |
| **Bahasa Pemrograman** | **Python** | 3.10+ / 3.13 | Bahasa komputasi utama untuk seluruh pipeline sains data dari hulu ke hilir. |
| **Penyedia Data Satelit** | **Copernicus CDSE API** | REST API v1 | Mengunduh nilai pantulan spektral 7 band Sentinel-2 Level-2A secara terprogram. |
| **Manipulasi Spasial** | **GeoPandas & Shapely** | Pustaka Python | Membaca geometri poligon sampel GeoJSON, memvalidasi koordinat WGS 84 (EPSG:4326). |
| **Pemrosesan Citra Raster** | **Rasterio** | Pustaka Python | Membaca file citra GeoTIFF Sentinel-2, pemotongan jendela (*windowed read*), dan ekspor raster tematik. |
| **Peta Interaktif Web** | **Leaflet.js & HTML5** | Web GIS Open-Source | Menyajikan visualisasi peta 2D interaktif, seleksi layer citra, dan filter dinamis tutupan lahan. |
| **Komputasi Numerik** | **Pandas & NumPy** | Pustaka Python | Membersihkan dataset tabular, mengaudit data, dan menghitung 5 indeks biofisik terarah. |
| **Machine Learning** | **Scikit-Learn** | Pustaka Python | Pembagian *Spatial Group Stratified Split*, normalisasi fitur, dan komputasi metrik evaluasi. |
| **Gradient Boosting** | **LightGBM** | Pustaka Python | Algoritma *boosting* pohon keputusan berkecepatan tinggi dengan optimasi parameter anti-overfitting. |
| **Serialisasi Model** | **Joblib** | Pustaka Python | Menyimpan dan memuat kembali artefak model terlatih (`.pkl`) untuk kebutuhan deployment. |
| **Visualisasi Saintifik** | **Matplotlib & Seaborn** | Pustaka Python | Menghasilkan grafik kurva spektral, heatmap matriks konfusi, kurva ROC-AUC, dan barplot fitur. |
| **Antarmuka Aplikasi Web** | **Streamlit** | Web Framework | Membangun dashboard analitik geospasial interaktif untuk pengujian dan evaluasi pengguna. |

---

## 1.8 Rencana Manajemen Risiko & Mitigasi

| Risiko Potensial | Dampak Teknis | Strategi Mitigasi Teruji |
| :--- | :--- | :--- |
| **Gangguan Tutupan Awan & Bayangan** | Nilai reflektansi BOA terdistorsi; awan putih disangka bangunan. | Filtering seleksi citra Sentinel-2 bebas awan (< 5%) dan modul deteksi masking awan terang pada pra-inferensi. |
| **Ambiguitas Spektral Antar Kelas** | Tertukarnya kelas mangrove vs hutan, sawah vs perairan, bangunan vs tanah terbuka. | Implementasi arsitektur **Mutual Disambiguation 5 Indeks Biofisik Terarah** (REMI, NDVI, IBI, MNDWI, NDWI). |
| **Kebocoran Spasial (*Spatial Leakage*)** | Akurasi model tampak tinggi palsu karena data latih dan uji berasal dari poligon yang berdekatan. | Menerapkan partisi data berbasis **Spatial Group Stratified Split 80:20** (pemisahan total level poligon acuan). |
| **Overfitting pada Data Latih** | Model menghafal sampel dan gagal generalisasi pada scene baru. | Regulasi ketat pada pohon LightGBM (`max_depth=6`, `num_leaves=20`, `min_child_samples=35`, subsampling, L1/L2 penalty). |
| **Overconfidence pada Piksel NoData** | Model memprediksi 100% perairan pada piksel hitam tepi citra (nilai 0). | Masking piksel invalid (NoData, saturasi, nilai di luar rentang latih) sebelum dan sesudah proses inferensi. |
| **Latensi Komputasi Super-Resolusi** | Pengguna mengalami freeze/hang saat memilih opsi resolusi detail (5m dan 2.5m). | Mengganti interpolasi scipy dengan **fast bilinear C-engine (PIL)** dan menerapkan *dynamic stride Leaflet overlay*. |

> **Kesimpulan Tahap Business Understanding:**  
> Dengan perumusan masalah yang presisi, strategi separasi dua arah 5 indeks biofisik, arsitektur model teratur anti-overfitting, serta integrasi deployment interaktif multi-resolusi (lokal & cloud), proyek sains data ini memiliki fondasi metodologis yang solid dan siap diimplementasikan secara komprehensif dari Tahap 2 hingga Tahap 6.

